# Step 1: Setup and Authentication

This cell installs necessary libraries, connects to your Google Drive (make sure you're logged in Colab and Drive with the same Google account), and enters your Google Gemini API Key. API Key needs to be acquired beforehand from Google AI Studio and stored in Colab secret variable GEMINI_API_KEY.

It needs to be run once at the beginning of the session and when the session is restarted.

In [ ]:
# Step 1: Setup and Authentication
# Run this cell to install necessary libraries, connect to your Google Drive, and enter your Gemini API Key.

# Install required Python libraries
!pip install google-genai pandas openpyxl pypdf -q

from google import genai
from google.colab import drive, userdata
import pandas as pd
import os
import re
from IPython.display import display, Markdown
import time
import datetime

# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# MOUNT GOOGLE DRIVE
# This allows the notebook to access your Google Drive files.
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
try:
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully.")
except Exception as e:
    print(f"❌ Error mounting Google Drive: {e}")

# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# CONFIGURE GEMINI API KEY
# You can get a key from https://aistudio.google.com/app/apikey
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
try:
    # Use Colab's user data feature for secure key storage
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except userdata.SecretNotFoundError:
    # Fallback to asking the user for the key if not found in secrets
    from google.colab import output
    output.enable_custom_widget_manager()
    from getpass import getpass
    GEMINI_API_KEY = getpass('Please enter your Google API Key: ')
    userdata.set('GEMINI_API_KEY', GEMINI_API_KEY)

try:
    client = genai.Client(api_key=GEMINI_API_KEY)
    print("✅ Gemini API client configured successfully.")
except Exception as e:
    print(f"❌ Error configuring Gemini API: {e}")
    print("   Please ensure you have entered a valid API key.")

# Step 2: Define File Paths and Prompt

This cell sets up prompt and creates two Excel files:
- output excel file with a given structure,
- excel file with processed file names.

It needs to be run at the beginning of the session and every time you change prompt, data subfolder or output structure.

This is the only part of the code that needs changing when the process is applied on new data.

You need to change the following:
- TOPIC - the subfolder where text data in pdf file format are stored
- PDF_FOLDER_PATH - the main folder that contains subfolders with data and where Excel file with results will be created
- EXCEL_FILE_PATH - The name of the output Excel file
- PROCESSED_FILES_PATH - The name of the excel file where the names of files already processed are stored
- GEMINI_MODEL - Selected Gemini model.
- GEMINI_PROMPT - Structured prompt with instructions for content analysis
- EXCEL_COLUMNS - The structure of output excel file. It needs to reflect the variables in the prompt.

In [ ]:
# Step 2: Define File Paths and Prompt
# Verify the folder and file names below, then run this cell.

# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# DEFINE FILE AND FOLDER PATHS
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
TOPIC = "LitReviewDemo"

# The folder inside your Google Drive containing the PDF files
PDF_FOLDER_PATH = os.path.join('/content/drive/MyDrive/LLMWorkflowDemo', TOPIC)

# The name of the output Excel file
EXCEL_FILE_PATH = '/content/drive/MyDrive/LLMWorkflowDemo/LLM Workflow Demo Results - Gemini.xlsx'

# Define the path for the processed files tracking sheet
PROCESSED_FILES_PATH = os.path.join(PDF_FOLDER_PATH, 'LLM Workflow Demo - Gemini - processed.xlsx')

GEMINI_MODEL = 'gemini-2.5-pro'
# GEMINI_MODEL = 'gemini-2.5-flash'
# GEMINI_MODEL = 'gemini-3-flash-preview'
# GEMINI_MODEL = 'gemini-3.1-pro-preview'

# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# DEFINE THE PROMPT FOR GEMINI
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
GEMINI_PROMPT = """
You are a researcher in Management conducting a literature review. You will receive a text of an academic paper. Please use the following coding manual to code the text.
●	IE_YEAR_PUBLISHED = The year the paper was published.
●	IE_JOURNAL = Journal in which the paper was published.
●	IE_AUTHORS = Authors of the paper.
●	IE_TITLE = Title of the paper.
●	IE_DOI = DOI of the paper.
●	AN_RQ = Did the authors explicitly define research questions, research aims, or research objectives? (1 = Yes, 0 = No).
●	SU_RQ_VERBATIM = Copy verbatim text from the paper that defines the research questions/aims/objectives. If none, return N/A.
●	SU_FULL_PROCESS = What were the steps of the research and analysis process from start to finish?
●	SU_SUMMARY = Summarize the paper in one paragraph.
●	SU_KEY_TAKEAWAYS = What are the key takeaways of the paper?
●	AN_PAPER_TYPE = Type of the paper – empirical, review, conceptual, editorial, other.
●	SU_PAPER_TYPE_REASON = Explain why you chose the paper type.
●	AN_METHOD_TYPE = What type of research methods were used (qualitative, quantitative, multiple, mixed)?
●	SU_METHOD_TYPE_REASON = Explain why you chose the research method type.
●	SU_METHOD = Describe research methods that were used in the paper.
●	SU_DATA_COLLECTION = How was the data collected?
●	SU_DATA_COLLECTION_VERBATIM = Copy verbatim text from the paper that describes data collection. If there are multiple relevant sections combine them (separated by “…”) in a single text. Return N/A if none.
●	IE_SAMPLE_SIZE = What was the size of the sample? Return N/A if not explicitly stated.
●	SU_KEY_FINDINGS = What were the key findings of the paper?
●	AN_THEORETICAL_CONTRIBUTION = Was the theoretical contribution explicitly described? (1 = Yes, 0 = No).
●	SU_THEORETICAL_CONTRIBUTION_VERBATIM = Copy verbatim text from the paper that describes the theoretical contribution of the study. If there are multiple relevant sections combine them (separated by “…”) in a single text. Return N/A if none.

Put all extracted variables in a table with columns: IE_YEAR_PUBLISHED, IE_JOURNAL, IE_AUTHORS, IE_TITLE, IE_DOI, AN_RQ, SU_RQ_VERBATIM, SU_FULL_PROCESS, SU_SUMMARY, SU_KEY_TAKEAWAYS,
AN_PAPER_TYPE, SU_PAPER_TYPE_REASON, AN_METHOD_TYPE, SU_METHOD_TYPE_REASON, SU_METHOD, SU_DATA_COLLECTION, SU_DATA_COLLECTION_VERBATIM, IE_SAMPLE_SIZE, SU_KEY_FINDINGS, AN_THEORETICAL_CONTRIBUTION,
SU_THEORETICAL_CONTRIBUTION_VERBATIM.

"""

# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# INITIALIZE EXCEL FILE
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# Define the columns for the Excel file
EXCEL_COLUMNS = [
    "IE_YEAR_PUBLISHED",
    "IE_JOURNAL",
    "IE_AUTHORS",
    "IE_TITLE",
    "IE_DOI",
    "AN_RQ",
    "SU_RQ_VERBATIM",
    "SU_FULL_PROCESS",
    "SU_SUMMARY",
    "SU_KEY_TAKEAWAYS",
    "AN_PAPER_TYPE",
    "SU_PAPER_TYPE_REASON",
    "AN_METHOD_TYPE",
    "SU_METHOD_TYPE_REASON",
    "SU_METHOD",
    "SU_DATA_COLLECTION",
    "SU_DATA_COLLECTION_VERBATIM",
    "IE_SAMPLE_SIZE",
    "SU_KEY_FINDINGS",
    "AN_THEORETICAL_CONTRIBUTION",
    "SU_THEORETICAL_CONTRIBUTION_VERBATIM",
    "Source file",
    "Topic",
    "Model",
    "Timestamp"
]

EXCELPROCESSED_COLUMNS = [
    "Filename"
]

# Create the Excel file with headers if it doesn't already exist
if not os.path.exists(EXCEL_FILE_PATH):
    df_initial = pd.DataFrame(columns=EXCEL_COLUMNS)
    df_initial.to_excel(EXCEL_FILE_PATH, index=False)
    print(f"✅ Created new Excel file at: {EXCEL_FILE_PATH}")
else:
    print(f"✅ Excel file already exists at: {EXCEL_FILE_PATH}. New data will be appended.")

# Create the Excel file with headers if it doesn't already exist
if not os.path.exists(PROCESSED_FILES_PATH):
    df_initial = pd.DataFrame(columns=EXCELPROCESSED_COLUMNS)
    df_initial.to_excel(PROCESSED_FILES_PATH, index=False)
    print(f"✅ Created new Excel file at: {PROCESSED_FILES_PATH}")
else:
    print(f"✅ Excel file already exists at: {PROCESSED_FILES_PATH}. New data will be appended.")

# Display configuration for user verification
display(Markdown(f"""
### Configuration Set:
* **PDF Folder:** `{PDF_FOLDER_PATH}`
* **Output Excel File:** `{EXCEL_FILE_PATH}`
* **Processed file list:** `{PROCESSED_FILES_PATH}`
* **Gemini Model:** `{GEMINI_MODEL}`
"""))

# Step 3: Process PDFs and Generate Report (markdown table version)

This cell loops through all pdf files in the TOPIC folder and calls the selected Gemini model with the specified prompt for each one of them. The results are then written in the output Excel file This may take a long time depending on the length & complexity of the prompt and the number and size of your PDF files.

If the file is successfully processed it's name is written in the processed files Excel so it is not processed for the second time if the cell is run again.

Some files occasionally aren't successfully processed for some reason. In these cases it is worth running this cell again to see if they will be processed in the second run.

In [ ]:
# Step 3: Process PDFs and Generate Report
# ▶️ Run this cell to start the extraction process.
# This may take a long time depending on the number and size of your PDF files.

# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# HELPER FUNCTION TO PARSE GEMINI'S RESPONSE
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
def parse_gemini_table(response_text, filename):
    """Parses markdown table text from Gemini into a list of dictionaries."""
    # Split the response into lines
    lines = response_text.strip().split('\n')

    # Filter out header separators like |---|---|
    data_lines = [line for line in lines if not re.match(r'^\s*\|---\|.*', line)]
    # Filter out blank lines or lines that aren't table rows
    data_lines = [line for line in data_lines if line.strip().startswith('|')]

    if len(data_lines) < 2: # Should have at least a header and one data row
        return []

    # The first valid line is assumed to be the header
    header_line = data_lines.pop(0)
    # Extract column names from the header row
    headers = [h.strip() for h in header_line.split('|') if h.strip()]

    # Process the rest of the lines as data
    parsed_rows = []
    for line in data_lines:
        cells = [c.strip() for c in line.split('|') if c.strip() != '']
        if len(cells) == len(headers):
            row_dict = dict(zip(headers, cells))
            parsed_rows.append(row_dict)

    return parsed_rows

# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# MAIN PROCESSING LOOP
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
def process_pdfs():
    """Main function to loop through PDFs, call Gemini, and save results."""

    # Check if the PDF folder exists
    if not os.path.isdir(PDF_FOLDER_PATH):
        print(f"❌ ERROR: The folder '{PDF_FOLDER_PATH}' was not found in your Google Drive.")
        return

    # Load or create the processed files DataFrame
    processed_files_df = pd.DataFrame(columns=['Filename'])
    if os.path.exists(PROCESSED_FILES_PATH):
        try:
            processed_files_df = pd.read_excel(PROCESSED_FILES_PATH)
            print(f"✅ Loaded processed files list from: {PROCESSED_FILES_PATH}")
        except Exception as e:
            print(f"❌ Error loading processed files list: {e}. Starting with an empty list.")

    # List all PDF files in the specified folder
    pdf_files = [f for f in os.listdir(PDF_FOLDER_PATH) if f.lower().endswith('.pdf')]

    if not pdf_files:
        print(f"❌ No PDF files found in '{PDF_FOLDER_PATH}'.")
        return

    # Filter out already processed files
    unprocessed_files = [f for f in pdf_files if f not in processed_files_df['Filename'].values]

    if not unprocessed_files:
        print("\n✅ All PDF files in the folder have already been processed.")
        return

    print(f"Found {len(pdf_files)} PDF file(s). {len(unprocessed_files)} unprocessed file(s) to process. Starting...")
    print("-" * 50)

    num_files = len(unprocessed_files)
    i = 1

    # Loop through each unprocessed PDF file
    for filename in unprocessed_files:
        full_path = os.path.join(PDF_FOLDER_PATH, filename)
        print(f"🔄 Processing {i}/{num_files}: {filename}...")

        try:
            # Upload the file to the Gemini API
            # uploaded_file = genai.upload_file(path=full_path, display_name=filename)
            uploaded_file = client.files.upload(file=full_path)
            print(f"   ...File uploaded successfully.")

            # Call the Gemini API with the file and the prompt
            response = client.models.generate_content(model=GEMINI_MODEL, contents=[GEMINI_PROMPT, uploaded_file])
            print(f"   ...AI analysis complete.")

            # Parse the markdown table from Gemini's response
            extracted_data = parse_gemini_table(response.text, filename)

            if not extracted_data:
                print(f"   ⚠️ WARNING: No structured data could be extracted from the AI response for {filename}.")
                # Mark as processed even if no data extracted to avoid reprocessing
                # new_processed_row = pd.DataFrame({'Filename': [filename]})
                # processed_files_df = pd.concat([processed_files_df, new_processed_row], ignore_index=True)
                # processed_files_df.to_excel(PROCESSED_FILES_PATH, index=False)
                # print(f"   ...Marked '{filename}' as processed (no data extracted).")
                continue

            # Convert the extracted data into a pandas DataFrame
            new_df = pd.DataFrame(extracted_data)

            # Ensure all required columns are present, fill missing ones with blanks
            for col in EXCEL_COLUMNS:
                if col not in new_df.columns:
                    new_df[col] = ''

            # Add the source filename and topic to the extracted data
            new_df['Source file'] = filename
            new_df['Topic'] = TOPIC
            new_df['Model'] = GEMINI_MODEL
            new_df['Timestamp'] = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            # Reorder columns to match the Excel file and keep only those columns
            new_df = new_df[EXCEL_COLUMNS]

            # Append the results to the main Excel file
            master_df = pd.read_excel(EXCEL_FILE_PATH)
            updated_df = pd.concat([master_df, new_df], ignore_index=True)
            updated_df.to_excel(EXCEL_FILE_PATH, index=False)

            print(f"   ✅ SUCCESS: Appended {len(new_df)} row(s) to '{os.path.basename(EXCEL_FILE_PATH)}'.")

            # Add the processed filename to the tracking sheet
            new_processed_row = pd.DataFrame({'Filename': [filename]})
            processed_files_df = pd.concat([processed_files_df, new_processed_row], ignore_index=True)
            processed_files_df.to_excel(PROCESSED_FILES_PATH, index=False)
            print(f"   ...Marked '{filename}' as processed.")


        except Exception as e:
            print(f"   ❌ ERROR processing {filename}: {e}")
            # Optional: Log errors to a file if needed

        finally:
            # Clean up the uploaded file from the API if it exists
            if 'uploaded_file' in locals() and uploaded_file:
                client.files.delete(name=uploaded_file.name)
            print("-" * 50)
            i = i + 1

        # Add a delay to avoid hitting rate limits
        # time.sleep(4)


    print("\n🎉 All unprocessed PDF files have been processed.")


# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# EXECUTE THE MAIN FUNCTION
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---

process_pdfs()


# Step 4 (Optional): View Final Report

This cell shows the contents of output Excel file.

In [ ]:
# Step 4: View Final Report
# Run this cell to display the contents of the final Excel file.

import pandas as pd
from IPython.display import display

try:
    final_df = pd.read_excel(EXCEL_FILE_PATH)
    print(f"Displaying the contents of '{os.path.basename(EXCEL_FILE_PATH)}':")
    display(final_df)
except FileNotFoundError:
    print(f"❌ The file '{EXCEL_FILE_PATH}' could not be found.")
except Exception as e:
    print(f"An error occurred while reading the Excel file: {e}")